In [ ]:
import os
import cv2
import pickle
from tqdm import tqdm
import numpy as np
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

In [ ]:
pickle_in = open("/content/drive/MyDrive/CVPR_practice/facePikle_X_train","rb")
X_train = pickle.load(pickle_in)

pickle_in = open("/content/drive/MyDrive/CVPR_practice/facePikle_Y_train","rb")
Y_train = pickle.load(pickle_in)

pickle_in = open("/content/drive/MyDrive/CVPR_practice/facePikle_X_test","rb")
X_test = pickle.load(pickle_in)

pickle_in = open("/content/drive/MyDrive/CVPR_practice/facePikle_Y_test","rb")
Y_test = pickle.load(pickle_in)

print(f"X_train= {X_train.shape} Y_train= {Y_train.shape}")
print(f"X_test= {X_test.shape} Y_test= {Y_test.shape}")

X_train= (901, 227, 227, 3) Y_train= (901,)
X_test= (225, 227, 227, 3) Y_test= (225,)


In [ ]:
pickle_in = open("/content/drive/MyDrive/CVPR_practice/all_data.pkl","rb")
all_data = pickle.load(pickle_in)


In [ ]:
X = np.array([item[0] for item in all_data])
y = np.array([item[1] for item in all_data])

In [ ]:
split = int(0.8 * len(X))

X_train = X[:split]
y_train = y[:split]

X_test = X[split:]
y_test = y[split:]

#print("Train images:", X_train.shape)
#print("Train labels:", y_train.shape)
#print("Test images :", X_test.shape)
#print("Test labels :", y_test.shape)

In [ ]:
#X_train = X_train.astype("float32") / 127.5 - 1
#X_test = X_test.astype("float32") / 127.5 - 1

In [ ]:
X_train = X_train/ 255.0
X_test  = X_test/ 255.0


In [ ]:


IMG_SIZE = 227
NUM_CLASSES = 75

inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))

# Data augmentation
x = layers.RandomFlip("horizontal")(inputs)
x = layers.RandomRotation(0.1)(x)
x = layers.RandomZoom(0.1)(x)
x = layers.RandomContrast(0.1)(x)

# Base model
base_model = keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights="imagenet"
)
base_model.trainable = False

x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(512, activation="relu")(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)

model = keras.Model(inputs, outputs)

model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()



/tmp/ipython-input-3514835798.py:12: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base_model = keras.applications.MobileNetV2(


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 227, 227, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_flip (RandomFlip)        │ (None, 227, 227, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_rotation                 │ (None, 227, 227, 3)    │             0 │
│ (RandomRotation)                │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_zoom (RandomZoom)        │ (None, 227, 227, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_contrast                 │ (None, 227, 227, 3)    │             0 │
│ (RandomContrast)                │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 8, 8, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 512)            │       655,872 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 512)            │         2,048 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 75)             │        38,475 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,954,379 (11.27 MB)

 Trainable params: 695,371 (2.65 MB)

 Non-trainable params: 2,259,008 (8.62 MB)

In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_accuracy",
        patience=6,
        restore_best_weights=True
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.3,
        patience=3
    )
]


In [ ]:
history = model.fit(
    X_train, Y_train,
    validation_split=0.2,
    epochs=20,
    batch_size=32,
    callbacks=callbacks,
    shuffle=True
)


Epoch 1/20
23/23 ━━━━━━━━━━━━━━━━━━━━ 53s 2s/step - accuracy: 0.1548 - loss: 4.3042 - val_accuracy: 0.3260 - val_loss: 2.8432 - learning_rate: 0.0010
Epoch 2/20
23/23 ━━━━━━━━━━━━━━━━━━━━ 84s 2s/step - accuracy: 0.7180 - loss: 1.2490 - val_accuracy: 0.6464 - val_loss: 1.7572 - learning_rate: 0.0010
Epoch 3/20
23/23 ━━━━━━━━━━━━━━━━━━━━ 55s 2s/step - accuracy: 0.8719 - loss: 0.6258 - val_accuracy: 0.7680 - val_loss: 1.3334 - learning_rate: 0.0010
Epoch 4/20
23/23 ━━━━━━━━━━━━━━━━━━━━ 74s 2s/step - accuracy: 0.9124 - loss: 0.3784 - val_accuracy: 0.8066 - val_loss: 1.0611 - learning_rate: 0.0010
Epoch 5/20
23/23 ━━━━━━━━━━━━━━━━━━━━ 44s 2s/step - accuracy: 0.9488 - loss: 0.2679 - val_accuracy: 0.8453 - val_loss: 0.8321 - learning_rate: 0.0010
Epoch 6/20
23/23 ━━━━━━━━━━━━━━━━━━━━ 44s 2s/step - accuracy: 0.9673 - loss: 0.2108 - val_accuracy: 0.8508 - val_loss: 0.7090 - learning_rate: 0.0010
Epoch 7/20
23/23 ━━━━━━━━━━━━━━━━━━━━ 47s 2s/step - accuracy: 0.9737 - loss: 0.1723 - val_accuracy: 

In [ ]:
model.save("mobilenetv2.keras")
